In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from tqdm import tqdm
import os

sc.settings.verbosity = 1
sns.set(font_scale=1)
sc.settings.set_figure_params(dpi=150)
sns.set_style("ticks")

from matplotlib import cm
from matplotlib.colors import ListedColormap

cm_color = cm.get_cmap("Reds", 128)
cm_grey = cm.get_cmap("Greys", 128)

Reds = ListedColormap(np.vstack((
    cm_grey(np.linspace(0.2, 0.2, 1)),
    cm_color(np.linspace(0.1, 1, 128)),
)))

In [ ]:
my_path="/home/felix/projects/facial/felix/data/reprocessed_data/"
# Create the save directory if it does not exist already
if not os.path.exists(my_path):
    os.makedirs(my_path)

In [ ]:
for dataset in ["C01939A4", "C01939A5", "C01939A6", "C01939E6"]:
    for bin_size in [30, 50]:
        adata = sc.read_h5ad(f"/home/felix/data/processed_face/stereoseq/{dataset}/raw/{bin_size}.h5ad")
        
        adata.raw = None
        
        adata.var["mt"] = adata.var_names.str.startswith("MT-")
        sc.pp.calculate_qc_metrics(adata, log1p=False, percent_top=None, qc_vars=["mt"], inplace=True)

        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)
        sc.pp.highly_variable_genes(adata, n_top_genes=2000)
        adata.obsm["X_pca"] = sc.tl.pca(
            sc.pp.scale(adata[:, adata.var.highly_variable].X, max_value=10),
            n_comps=30,
            zero_center=True,
        )

        sc.pp.neighbors(adata, n_neighbors=20)
        sc.tl.umap(adata)
        for resolution in [0.25, 0.5, 1]:
            sc.tl.leiden(adata, resolution=resolution, key_added=f"leiden_{resolution}")

        obsp_keys = list(adata.obsp.keys())
        for obsp in obsp_keys:
            del adata.obsp[obsp]
        varm_keys = list(adata.varm.keys())
        for varm in varm_keys:
            del adata.obsp[varm]

        adata.var = pd.DataFrame(index=adata.var_names)
        del adata.obs["x"], adata.obs["y"], adata.obs["orig.ident"]

        adata.obsm["X_spatial"] = adata.obsm["spatial"].copy()
        del adata.obsm["spatial"]

        
        adata.write_h5ad(f"{my_path}{dataset}/processed/for_morans_{bin_size}.h5ad")
        

In [ ]:
for dataset in ["A02989E1", "A02993E1", "A02994D6", "A02994E6", "A02996D4"]:
    for bin_size in [30, 50]:
        adata = sc.read_h5ad(f"/home/felix/data/processed_face/stereoseq/{dataset}/raw/{bin_size}.h5ad")
        adata.raw = None
        
        adata.var["mt"] = adata.var_names.str.startswith("MT-")
        sc.pp.calculate_qc_metrics(adata, log1p=False, percent_top=None, qc_vars=["mt"], inplace=True)

        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)
        sc.pp.highly_variable_genes(adata, n_top_genes=2000)
        adata.obsm["X_pca"] = sc.tl.pca(
            sc.pp.scale(adata[:, adata.var.highly_variable].X, max_value=10),
            n_comps=30,
            zero_center=True,
        )

        sc.pp.neighbors(adata, n_neighbors=20)
        sc.tl.umap(adata)
        for resolution in [0.25, 0.5, 1]:
            sc.tl.leiden(adata, resolution=resolution, key_added=f"leiden_{resolution}")

        obsp_keys = list(adata.obsp.keys())
        for obsp in obsp_keys:
            del adata.obsp[obsp]
        varm_keys = list(adata.varm.keys())
        for varm in varm_keys:
            del adata.obsp[varm]

        adata.var = pd.DataFrame(index=adata.var_names)
        del adata.obs["x"], adata.obs["y"], adata.obs["orig.ident"]

        adata.obsm["X_spatial"] = adata.obsm["spatial"].copy()
        del adata.obsm["spatial"]

        adata.write_h5ad(f"{my_path}{dataset}/processed/for_morans_{bin_size}.h5ad")